# DR9 Radius-Grid Local Overdensity Visualization

This notebook reads the output of `DR9_radius_grid_overdensity_mpi.py` and tests which signal/background radius definition gives the strongest relation between local environment and cluster richness.

The two richness columns of interest are:

- `LAMBDA`: redMaPPer richness
- `lambda_spec_tot`: spectroscopic richness

The fiducial local-environment statistic is `Sigma_excess`, the coverage-corrected signal surface density minus the coverage-corrected background surface density.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy.stats import binned_statistic, pearsonr, rankdata, spearmanr

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == 'local_overdensity':
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

OUTPUT_DIR = REPO_ROOT / 'local_overdensity' / 'dr9_outputs' / 'radius_grid_overdensity'
PLOT_DIR = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_TABLE = OUTPUT_DIR / 'rm_dr9_radius_grid_overdensity.fits'
INPUT_SUMMARY = OUTPUT_DIR / 'rm_dr9_radius_grid_overdensity_summary.ecsv'

print('Input table:', INPUT_TABLE)
print('Input summary:', INPUT_SUMMARY)
print('Plot dir:', PLOT_DIR)

In [ ]:
# Configuration
RICHNESS_COLS = ['LAMBDA', 'lambda_spec_tot']
Y_COLS = ['Sigma_excess', 'N_excess', 'Sigma_signal_covcorr', 'Sigma_background_covcorr']
FIDUCIAL_Y_COL = 'Sigma_excess'
NBINS = 8
COVERAGE_MIN_FOR_PLOTS = 0.0  # set to 0.8 for a stricter sanity cut if desired
N_BOOT = 1000
BOOTSTRAP_SEED = 12345
Z_SPLIT_BINS = [(0.1, 0.33), (0.33, 0.4)]

## Load MPI Output

In [ ]:
results = Table.read(INPUT_TABLE)
summary = Table.read(INPUT_SUMMARY)

# Compatibility with older/small test outputs: derive this ranking column if needed.
if 'abs_spearman_r' not in summary.colnames:
    spearman_r = np.ma.asarray(summary['spearman_r'], dtype=float)
    summary['abs_spearman_r'] = np.abs(np.ma.filled(spearman_r, np.nan))

print(f'Result rows: {len(results):,}')
print(f'Summary rows: {len(summary):,}')
print(results.colnames)
summary[:10]

## Helper Functions

In [ ]:
def col_float(table, col):
    arr = np.ma.asarray(table[col], dtype=float)
    return np.ma.filled(arr, np.nan)


def finite_positive(x):
    x = np.asarray(x, dtype=float)
    return np.isfinite(x) & (x > 0)


def correlation_summary(x, y, mask=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if mask is None:
        mask = np.ones(len(x), dtype=bool)
    mask = np.asarray(mask, dtype=bool) & finite_positive(x) & np.isfinite(y)
    if np.count_nonzero(mask) < 4:
        return np.nan, np.nan, np.nan, np.nan, np.count_nonzero(mask)
    sr, sp = spearmanr(x[mask], y[mask])
    pr, pp = pearsonr(np.log10(x[mask]), y[mask])
    return sr, sp, pr, pp, np.count_nonzero(mask)


def bootstrap_correlations(x, y, mask=None, n_boot=1000, seed=12345):
    """Return Pearson/Spearman coefficients with bootstrap 16-84% intervals."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if mask is None:
        mask = np.ones(len(x), dtype=bool)
    mask = np.asarray(mask, dtype=bool) & finite_positive(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    n = len(x)
    if n < 4:
        return {
            'N_corr': n,
            'spearman_r': np.nan,
            'spearman_p': np.nan,
            'spearman_lo': np.nan,
            'spearman_hi': np.nan,
            'spearman_err_low': np.nan,
            'spearman_err_high': np.nan,
            'pearson_r_logx': np.nan,
            'pearson_p_logx': np.nan,
            'pearson_lo': np.nan,
            'pearson_hi': np.nan,
            'pearson_err_low': np.nan,
            'pearson_err_high': np.nan,
        }

    spearman_r, spearman_p = spearmanr(x, y)
    pearson_r, pearson_p = pearsonr(np.log10(x), y)

    spearman_boot = np.full(n_boot, np.nan)
    pearson_boot = np.full(n_boot, np.nan)
    logx = np.log10(x)
    for iboot in range(n_boot):
        ind = rng.integers(0, n, size=n)
        if len(np.unique(x[ind])) > 1 and len(np.unique(y[ind])) > 1:
            spearman_boot[iboot] = spearmanr(x[ind], y[ind]).statistic
            pearson_boot[iboot] = pearsonr(logx[ind], y[ind]).statistic

    sp_lo, sp_hi = np.nanpercentile(spearman_boot, [16, 84])
    pr_lo, pr_hi = np.nanpercentile(pearson_boot, [16, 84])
    return {
        'N_corr': n,
        'spearman_r': float(spearman_r),
        'spearman_p': float(spearman_p),
        'spearman_lo': float(sp_lo),
        'spearman_hi': float(sp_hi),
        'spearman_err_low': float(spearman_r - sp_lo),
        'spearman_err_high': float(sp_hi - spearman_r),
        'pearson_r_logx': float(pearson_r),
        'pearson_p_logx': float(pearson_p),
        'pearson_lo': float(pr_lo),
        'pearson_hi': float(pr_hi),
        'pearson_err_low': float(pearson_r - pr_lo),
        'pearson_err_high': float(pr_hi - pearson_r),
    }


def bootstrap_summary_from_results(results, richness_cols, y_cols, z_mask=None, n_boot=1000, seed=12345):
    rows = []
    radius_ids = np.unique(np.asarray(results['radius_def_id'], dtype=int))
    for radius_id in radius_ids:
        tab = results[np.asarray(results['radius_def_id'], dtype=int) == radius_id]
        cov_sig = col_float(tab, 'coverage_signal')
        cov_bg = col_float(tab, 'coverage_background')
        base_mask = np.isfinite(cov_sig) & np.isfinite(cov_bg) & (cov_sig >= COVERAGE_MIN_FOR_PLOTS) & (cov_bg >= COVERAGE_MIN_FOR_PLOTS)
        if z_mask is not None:
            base_mask &= z_mask(tab)
        for richness_col in richness_cols:
            x = col_float(tab, richness_col)
            for y_col in y_cols:
                y = col_float(tab, y_col)
                stats = bootstrap_correlations(
                    x,
                    y,
                    mask=base_mask,
                    n_boot=n_boot,
                    seed=seed + 1009 * int(radius_id) + 37 * len(rows),
                )
                first = tab[0]
                rows.append({
                    'radius_def_id': int(radius_id),
                    'radius_label': str(first['radius_label']),
                    'r_sig_in_hmpc': float(first['r_sig_in_hmpc']),
                    'r_sig_out_hmpc': float(first['r_sig_out_hmpc']),
                    'r_bg_in_hmpc': float(first['r_bg_in_hmpc']),
                    'r_bg_out_hmpc': float(first['r_bg_out_hmpc']),
                    'richness_col': richness_col,
                    'y_col': y_col,
                    **stats,
                })
    out = pd.DataFrame(rows)
    out['abs_spearman_r'] = np.abs(out['spearman_r'])
    return out


def add_binned_mean(ax, x, y, nbins=8, color='crimson', label='binned mean'):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = finite_positive(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < nbins:
        return

    bins = np.logspace(np.log10(np.nanmin(x)), np.log10(np.nanmax(x)), nbins + 1)
    centers = np.sqrt(bins[:-1] * bins[1:])
    mean, _, _ = binned_statistic(x, y, statistic='mean', bins=bins)
    std, _, _ = binned_statistic(x, y, statistic='std', bins=bins)
    count, _, _ = binned_statistic(x, y, statistic='count', bins=bins)
    good = count > 3
    sem = std / np.sqrt(np.clip(count, 1, None))
    ax.errorbar(centers[good], mean[good], yerr=sem[good], fmt='o', color=color,
                ecolor=color, capsize=3, label=label, zorder=5)


def add_binned_mean_linear_x(ax, x, y, nbins=8, color='crimson', label='binned mean'):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < nbins:
        return

    bins = np.linspace(np.nanmin(x), np.nanmax(x), nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean, _, _ = binned_statistic(x, y, statistic='mean', bins=bins)
    std, _, _ = binned_statistic(x, y, statistic='std', bins=bins)
    count, _, _ = binned_statistic(x, y, statistic='count', bins=bins)
    good = count > 3
    sem = std / np.sqrt(np.clip(count, 1, None))
    ax.errorbar(centers[good], mean[good], yerr=sem[good], fmt='o', color=color,
                ecolor=color, capsize=3, label=label, zorder=5)


def label_for_radius(row):
    return (
        rf'$S:[{row["r_sig_in_hmpc"]:g},{row["r_sig_out_hmpc"]:g}]$ '
        rf'$B:[{row["r_bg_in_hmpc"]:g},{row["r_bg_out_hmpc"]:g}]$'
    )

## Rank Radius Definitions

The summary table is generated by the MPI script. Here we sort it for each richness and environment statistic. The default ranking is by largest absolute Spearman coefficient.

In [ ]:
# Compute bootstrap errors directly from the per-cluster results table.
# This can take a little while if N_BOOT is large; reduce N_BOOT for quick inspection.
bootstrap_df = bootstrap_summary_from_results(
    results,
    richness_cols=RICHNESS_COLS,
    y_cols=Y_COLS,
    n_boot=N_BOOT,
    seed=BOOTSTRAP_SEED,
)
summary_df = bootstrap_df.sort_values('abs_spearman_r', ascending=False)

base_display_cols = [
    'radius_def_id', 'radius_label', 'r_sig_in_hmpc', 'r_sig_out_hmpc',
    'r_bg_in_hmpc', 'r_bg_out_hmpc', 'N_corr',
    'spearman_r', 'spearman_err_low', 'spearman_err_high', 'spearman_p',
    'pearson_r_logx', 'pearson_err_low', 'pearson_err_high', 'pearson_p_logx',
]

for richness_col in RICHNESS_COLS:
    print('\n' + '=' * 80)
    print(richness_col)
    sub = summary_df[
        (summary_df['richness_col'] == richness_col)
        & (summary_df['y_col'] == FIDUCIAL_Y_COL)
    ].sort_values('abs_spearman_r', ascending=False)
    display(sub[base_display_cols])

## Correlation Strength Versus Radius Definition

In [ ]:
fig, axes = plt.subplots(len(RICHNESS_COLS), 1, figsize=(11, 4.2 * len(RICHNESS_COLS)), sharex=True)
if len(RICHNESS_COLS) == 1:
    axes = [axes]

for ax, richness_col in zip(axes, RICHNESS_COLS):
    sub = summary_df[
        (summary_df['richness_col'] == richness_col)
        & (summary_df['y_col'] == FIDUCIAL_Y_COL)
    ].sort_values('radius_def_id')
    x = np.arange(len(sub))
    labels = [
        f"S:{a:g}-{b:g}\nB:{c:g}-{d:g}"
        for a, b, c, d in zip(
            sub['r_sig_in_hmpc'], sub['r_sig_out_hmpc'],
            sub['r_bg_in_hmpc'], sub['r_bg_out_hmpc']
        )
    ]
    ax.axhline(0.0, color='black', lw=1, alpha=0.45)
    ax.errorbar(
        x,
        sub['spearman_r'],
        yerr=[sub['spearman_err_low'], sub['spearman_err_high']],
        marker='o',
        ls='-',
        capsize=3,
        color='crimson',
        label='Spearman',
    )
    ax.errorbar(
        x,
        sub['pearson_r_logx'],
        yerr=[sub['pearson_err_low'], sub['pearson_err_high']],
        marker='s',
        ls='-',
        capsize=3,
        color='royalblue',
        label=r'Pearson $(\log x,y)$',
    )
    best = sub.iloc[np.nanargmax(np.abs(sub['spearman_r']))]
    best_x = np.where(sub['radius_def_id'].values == best['radius_def_id'])[0][0]
    ax.scatter(best_x, best['spearman_r'], s=110, facecolor='none', edgecolor='black', lw=1.6, zorder=6)
    ax.set_ylabel('correlation coefficient')
    ax.set_title(f'{FIDUCIAL_Y_COL} versus {richness_col}')
    ax.legend(frameon=False)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=0, fontsize=9)

axes[-1].set_xlabel(r'radius definition [$h^{-1}{\rm Mpc}$]')
fig.tight_layout()
fig.savefig(PLOT_DIR / f'correlation_vs_radius_definition_{FIDUCIAL_Y_COL}_bootstrap_errors.png', dpi=180)
plt.show()

## Redshift-Split Correlation Versus Radius For `lambda_spec_tot`

This computes the Pearson and Spearman correlations between `lambda_spec_tot` and the fiducial environment statistic in two redshift bins, `0.1<z<0.33` and `0.33<z<0.4`. Error bars are bootstrap 16-84% intervals from resampling clusters within each redshift bin.

In [ ]:
Z_COL = 'Z_SPEC_x' if 'Z_SPEC_x' in results.colnames else ('Z_SPEC' if 'Z_SPEC' in results.colnames else 'Z_LAMBDA')

fig, axes = plt.subplots(1, len(Z_SPLIT_BINS), figsize=(7.2 * len(Z_SPLIT_BINS), 4.9), sharey=True)
if len(Z_SPLIT_BINS) == 1:
    axes = [axes]

z_split_rows = []
for ax, (zlo, zhi) in zip(axes, Z_SPLIT_BINS):
    def this_z_mask(tab, zlo=zlo, zhi=zhi):
        z = col_float(tab, Z_COL)
        return (z >= zlo) & (z < zhi)

    z_df = bootstrap_summary_from_results(
        results,
        richness_cols=['lambda_spec_tot'],
        y_cols=[FIDUCIAL_Y_COL],
        z_mask=this_z_mask,
        n_boot=N_BOOT,
        seed=BOOTSTRAP_SEED + int(1000 * zlo),
    ).sort_values('radius_def_id')
    z_df['z_low'] = zlo
    z_df['z_high'] = zhi
    z_split_rows.append(z_df)

    x = np.arange(len(z_df))
    labels = [
        f"S:{a:g}-{b:g}\nB:{c:g}-{d:g}"
        for a, b, c, d in zip(
            z_df['r_sig_in_hmpc'], z_df['r_sig_out_hmpc'],
            z_df['r_bg_in_hmpc'], z_df['r_bg_out_hmpc']
        )
    ]
    ax.axhline(0.0, color='black', lw=1, alpha=0.45)
    ax.errorbar(
        x,
        z_df['spearman_r'],
        yerr=[z_df['spearman_err_low'], z_df['spearman_err_high']],
        marker='o',
        ls='-',
        capsize=3,
        color='crimson',
        label='Spearman',
    )
    ax.errorbar(
        x,
        z_df['pearson_r_logx'],
        yerr=[z_df['pearson_err_low'], z_df['pearson_err_high']],
        marker='s',
        ls='-',
        capsize=3,
        color='royalblue',
        label=r'Pearson $(\log\lambda_{\rm spec}, y)$',
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(rf'${zlo}<z<{zhi}$')
    ax.set_xlabel(r'radius definition [$h^{-1}{\rm Mpc}$]')
    ax.legend(frameon=False)

axes[0].set_ylabel('correlation coefficient')
fig.suptitle(rf'Correlation of {FIDUCIAL_Y_COL} with $\lambda_{{\rm spec,tot}}$ by redshift', fontsize=14)
fig.tight_layout()
fig.savefig(PLOT_DIR / f'lambda_spec_correlation_vs_radius_redshift_split_{FIDUCIAL_Y_COL}.png', dpi=180)
plt.show()

z_split_summary = pd.concat(z_split_rows, ignore_index=True)
z_split_summary.to_csv(PLOT_DIR / f'lambda_spec_correlation_vs_radius_redshift_split_{FIDUCIAL_Y_COL}.csv', index=False)
display(z_split_summary[[
    'z_low', 'z_high', 'radius_def_id', 'radius_label', 'N_corr',
    'spearman_r', 'spearman_err_low', 'spearman_err_high', 'spearman_p',
    'pearson_r_logx', 'pearson_err_low', 'pearson_err_high', 'pearson_p_logx'
]])

## Optional Heatmap View

This is most useful if the MPI script was run with `--pair-mode all`, producing a grid of signal and background choices.

In [ ]:
for richness_col in RICHNESS_COLS:
    sub = summary_df[
        (summary_df['richness_col'] == richness_col)
        & (summary_df['y_col'] == FIDUCIAL_Y_COL)
    ].copy()
    sub['sig_label'] = sub['r_sig_in_hmpc'].astype(str) + '-' + sub['r_sig_out_hmpc'].astype(str)
    sub['bg_label'] = sub['r_bg_in_hmpc'].astype(str) + '-' + sub['r_bg_out_hmpc'].astype(str)
    pivot = sub.pivot_table(index='sig_label', columns='bg_label', values='spearman_r', aggfunc='first')

    fig, ax = plt.subplots(figsize=(1.2 * max(5, pivot.shape[1]), 0.8 * max(4, pivot.shape[0])))
    im = ax.imshow(pivot.values, origin='lower', cmap='coolwarm', aspect='auto')
    ax.set_xticks(np.arange(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(pivot.shape[0]))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel(r'background annulus [$h^{-1}{\rm Mpc}$]')
    ax.set_ylabel(r'signal annulus [$h^{-1}{\rm Mpc}$]')
    ax.set_title(f'Spearman r: {FIDUCIAL_Y_COL} versus {richness_col}')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Spearman r')
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'heatmap_spearman_{richness_col}_{FIDUCIAL_Y_COL}.png', dpi=180)
    plt.show()

## Best-Radius Binned Mean And Rank-Rank Plots

For each richness definition, this takes the radius definition with the largest \(|r_s|\) for `Sigma_excess` and visualizes both the raw binned mean and the rank-rank relation.

In [ ]:
for richness_col in RICHNESS_COLS:
    sub = summary_df[
        (summary_df['richness_col'] == richness_col)
        & (summary_df['y_col'] == FIDUCIAL_Y_COL)
    ].sort_values('abs_spearman_r', ascending=False)
    best = sub.iloc[0]
    radius_id = int(best['radius_def_id'])

    tab = results[results['radius_def_id'] == radius_id]
    x = col_float(tab, richness_col)
    y = col_float(tab, FIDUCIAL_Y_COL)
    cov_sig = col_float(tab, 'coverage_signal')
    cov_bg = col_float(tab, 'coverage_background')
    mask = finite_positive(x) & np.isfinite(y) & (cov_sig >= COVERAGE_MIN_FOR_PLOTS) & (cov_bg >= COVERAGE_MIN_FOR_PLOTS)

    sr, sp, pr, pp, n = correlation_summary(x, y, mask)

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
    axes[0].scatter(x[mask], y[mask], s=12, alpha=0.35, color='0.25', edgecolor='none')
    add_binned_mean(axes[0], x[mask], y[mask], nbins=NBINS, color='crimson')
    axes[0].set_xscale('log')
    axes[0].set_xlabel(richness_col)
    axes[0].set_ylabel(FIDUCIAL_Y_COL)
    axes[0].set_title(label_for_radius(best))
    axes[0].text(
        0.04, 0.96,
        rf'$N={n}$' + '\n' + rf'Spearman $r_s={sr:.3f}$, $p={sp:.1e}$' + '\n' + rf'Pearson $r={pr:.3f}$, $p={pp:.1e}$',
        transform=axes[0].transAxes,
        ha='left', va='top', fontsize=10,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8),
    )
    axes[0].legend(frameon=False)

    rx = rankdata(x[mask])
    ry = rankdata(y[mask])
    axes[1].scatter(rx, ry, s=12, alpha=0.35, color='0.25', edgecolor='none')
    add_binned_mean_linear_x(axes[1], rx, ry, nbins=NBINS, color='crimson')
    axes[1].plot([np.min(rx), np.max(rx)], [np.min(rx), np.max(rx)], color='black', lw=1, alpha=0.35)
    axes[1].set_xlabel(f'rank({richness_col})')
    axes[1].set_ylabel(f'rank({FIDUCIAL_Y_COL})')
    axes[1].set_title('Rank-rank visualization')

    fig.suptitle(f'Best radius definition for {richness_col}', fontsize=14)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'best_radius_binned_rank_{richness_col}_{FIDUCIAL_Y_COL}.png', dpi=180)
    plt.show()

## Redshift-Binned Check

This checks whether the preferred correlation survives within redshift bins, rather than being driven mainly by redshift evolution or angular-size effects.

In [ ]:
Z_COL = 'Z_SPEC_x' if 'Z_SPEC_x' in results.colnames else ('Z_SPEC' if 'Z_SPEC' in results.colnames else None)
Z_BINS = [(0.1, 0.2), (0.2, 0.3), (0.3, 0.4)]

if Z_COL is None:
    print('No redshift column found.')
else:
    for richness_col in RICHNESS_COLS:
        sub = summary_df[
            (summary_df['richness_col'] == richness_col)
            & (summary_df['y_col'] == FIDUCIAL_Y_COL)
        ].sort_values('abs_spearman_r', ascending=False)
        best = sub.iloc[0]
        radius_id = int(best['radius_def_id'])
        tab = results[results['radius_def_id'] == radius_id]

        x = col_float(tab, richness_col)
        y = col_float(tab, FIDUCIAL_Y_COL)
        z = col_float(tab, Z_COL)
        cov_sig = col_float(tab, 'coverage_signal')
        cov_bg = col_float(tab, 'coverage_background')

        fig, axes = plt.subplots(1, len(Z_BINS), figsize=(16, 4.8), sharex=True, sharey=True)
        for ax, (zlo, zhi) in zip(axes, Z_BINS):
            zmask = (z >= zlo) & (z < zhi)
            mask = finite_positive(x) & np.isfinite(y) & zmask & (cov_sig >= COVERAGE_MIN_FOR_PLOTS) & (cov_bg >= COVERAGE_MIN_FOR_PLOTS)
            sr, sp, pr, pp, n = correlation_summary(x, y, mask)
            ax.scatter(x[mask], y[mask], s=12, alpha=0.4, color='0.25', edgecolor='none')
            add_binned_mean(ax, x[mask], y[mask], nbins=NBINS, color='crimson')
            ax.set_xscale('log')
            ax.set_title(rf'${zlo}<z<{zhi}$, $N={n}$' + '\n' + rf'$r_s={sr:.2f}$, $p={sp:.1e}$')
            ax.set_xlabel(richness_col)
        axes[0].set_ylabel(FIDUCIAL_Y_COL)
        fig.suptitle(f'Redshift-binned check: {richness_col}, {label_for_radius(best)}', fontsize=14)
        fig.tight_layout()
        fig.savefig(PLOT_DIR / f'redshift_binned_{richness_col}_{FIDUCIAL_Y_COL}.png', dpi=180)
        plt.show()

## Diagnose Zero Excess Surface Density

This section classifies rows with `Sigma_excess` consistent with zero. A zero excess can happen for several different reasons: the cluster has no valid covered area, the signal/background counts are both zero, the coverage-corrected signal and background densities are numerically equal, or the selected DR9 magnitude cut is so bright that the annuli are sparsely populated.

In [ ]:
ZERO_TOL = 1.0e-12

# Choose which radius definition to inspect. By default, use the best lambda_spec_tot radius.
best_lambda_spec = summary_df[
    (summary_df['richness_col'] == 'lambda_spec_tot')
    & (summary_df['y_col'] == FIDUCIAL_Y_COL)
].sort_values('abs_spearman_r', ascending=False).iloc[0]
DIAG_RADIUS_DEF_ID = int(best_lambda_spec['radius_def_id'])

# Set to None to diagnose all radius definitions together.
# DIAG_RADIUS_DEF_ID = None

if DIAG_RADIUS_DEF_ID is None:
    diag = results.copy()
    radius_label = 'all radius definitions'
else:
    diag = results[np.asarray(results['radius_def_id'], dtype=int) == DIAG_RADIUS_DEF_ID]
    radius_label = str(diag['radius_label'][0])

sigma_excess = col_float(diag, 'Sigma_excess')
sigma_signal = col_float(diag, 'Sigma_signal_covcorr')
sigma_bg = col_float(diag, 'Sigma_background_covcorr')
n_signal = col_float(diag, 'N_signal')
n_bg = col_float(diag, 'N_background')
area_signal = col_float(diag, 'area_signal_deg2')
area_bg = col_float(diag, 'area_background_deg2')
covered_signal = col_float(diag, 'covered_area_signal_deg2')
covered_bg = col_float(diag, 'covered_area_background_deg2')
coverage_signal = col_float(diag, 'coverage_signal')
coverage_bg = col_float(diag, 'coverage_background')
n_touch = col_float(diag, 'n_files_touching_cluster')

finite_excess = np.isfinite(sigma_excess)
zero_excess = finite_excess & (np.abs(sigma_excess) <= ZERO_TOL)
near_zero_excess = finite_excess & (np.abs(sigma_excess) <= np.nanpercentile(np.abs(sigma_excess[finite_excess]), 5))

categories = {
    'all rows': np.ones(len(diag), dtype=bool),
    'finite Sigma_excess': finite_excess,
    'exact/near numerical zero': zero_excess,
    'bottom 5% |Sigma_excess|': near_zero_excess,
    'no touching sweep file': n_touch <= 0,
    'zero covered signal area': ~np.isfinite(covered_signal) | (covered_signal <= 0),
    'zero covered background area': ~np.isfinite(covered_bg) | (covered_bg <= 0),
    'zero signal count': np.isfinite(n_signal) & (n_signal == 0),
    'zero background count': np.isfinite(n_bg) & (n_bg == 0),
    'zero signal and background counts': (np.isfinite(n_signal) & (n_signal == 0) & np.isfinite(n_bg) & (n_bg == 0)),
    'equal finite densities': np.isfinite(sigma_signal) & np.isfinite(sigma_bg) & (np.abs(sigma_signal - sigma_bg) <= ZERO_TOL),
}

diag_rows = []
for name, mask in categories.items():
    diag_rows.append({
        'category': name,
        'N': int(np.count_nonzero(mask)),
        'fraction': float(np.count_nonzero(mask) / len(diag)) if len(diag) else np.nan,
        'N_also_zero_excess': int(np.count_nonzero(mask & zero_excess)),
    })

diag_summary = pd.DataFrame(diag_rows)
print('Diagnosing:', radius_label)
display(diag_summary)

# Extra table: among zero-excess rows only, show why they are zero.
zero_cause_rows = []
if np.count_nonzero(zero_excess) > 0:
    zero_cause_masks = {
        'zero signal and bg counts': categories['zero signal and background counts'],
        'zero signal count only': categories['zero signal count'] & ~categories['zero background count'],
        'zero bg count only': categories['zero background count'] & ~categories['zero signal count'],
        'zero/invalid signal covered area': categories['zero covered signal area'],
        'zero/invalid bg covered area': categories['zero covered background area'],
        'equal finite densities': categories['equal finite densities'],
    }
    for name, mask in zero_cause_masks.items():
        zero_cause_rows.append({
            'zero_excess_cause': name,
            'N': int(np.count_nonzero(zero_excess & mask)),
            'fraction_of_zero_excess': float(np.count_nonzero(zero_excess & mask) / np.count_nonzero(zero_excess)),
        })
    display(pd.DataFrame(zero_cause_rows))
else:
    print('No exact zero-excess rows with current ZERO_TOL.')

### Zero-Excess Diagnostic Plots

These plots check whether zero-excess clusters are concentrated in redshift, richness, sky position, or coverage/count space.

In [ ]:
RA_COL = 'RA_x' if 'RA_x' in diag.colnames else ('RA' if 'RA' in diag.colnames else None)
DEC_COL = 'DEC_x' if 'DEC_x' in diag.colnames else ('DEC' if 'DEC' in diag.colnames else None)
Z_COL_DIAG = 'Z_SPEC_x' if 'Z_SPEC_x' in diag.colnames else ('Z_SPEC' if 'Z_SPEC' in diag.colnames else 'Z_LAMBDA')

z = col_float(diag, Z_COL_DIAG)
lambda_rm = col_float(diag, 'LAMBDA')
lambda_spec = col_float(diag, 'lambda_spec_tot')

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

axes[0].hist(z[finite_excess], bins=30, alpha=0.55, label='finite excess')
axes[0].hist(z[zero_excess], bins=30, alpha=0.75, label='zero excess')
axes[0].set_xlabel(Z_COL_DIAG)
axes[0].set_ylabel('clusters')
axes[0].legend(frameon=False)

axes[1].hist(lambda_rm[finite_excess & finite_positive(lambda_rm)], bins=30, alpha=0.55, label='finite excess')
axes[1].hist(lambda_rm[zero_excess & finite_positive(lambda_rm)], bins=30, alpha=0.75, label='zero excess')
axes[1].set_xlabel('LAMBDA')
axes[1].set_xscale('log')
axes[1].legend(frameon=False)

axes[2].hist(lambda_spec[finite_excess & finite_positive(lambda_spec)], bins=30, alpha=0.55, label='finite excess')
axes[2].hist(lambda_spec[zero_excess & finite_positive(lambda_spec)], bins=30, alpha=0.75, label='zero excess')
axes[2].set_xlabel('lambda_spec_tot')
axes[2].set_xscale('log')
axes[2].legend(frameon=False)

axes[3].scatter(coverage_signal, coverage_bg, s=10, alpha=0.25, color='0.4', label='all')
axes[3].scatter(coverage_signal[zero_excess], coverage_bg[zero_excess], s=16, alpha=0.8, color='crimson', label='zero excess')
axes[3].set_xlabel('coverage_signal')
axes[3].set_ylabel('coverage_background')
axes[3].legend(frameon=False)

axes[4].scatter(n_signal, n_bg, s=10, alpha=0.25, color='0.4', label='all')
axes[4].scatter(n_signal[zero_excess], n_bg[zero_excess], s=16, alpha=0.8, color='crimson', label='zero excess')
axes[4].set_xlabel('N_signal')
axes[4].set_ylabel('N_background')
axes[4].set_xscale('symlog', linthresh=1)
axes[4].set_yscale('symlog', linthresh=1)
axes[4].legend(frameon=False)

if RA_COL is not None and DEC_COL is not None:
    ra = col_float(diag, RA_COL)
    dec = col_float(diag, DEC_COL)
    axes[5].scatter(ra[finite_excess], dec[finite_excess], s=5, alpha=0.2, color='0.4', label='finite excess')
    axes[5].scatter(ra[zero_excess], dec[zero_excess], s=10, alpha=0.8, color='crimson', label='zero excess')
    axes[5].set_xlabel(RA_COL)
    axes[5].set_ylabel(DEC_COL)
    axes[5].legend(frameon=False)
else:
    axes[5].axis('off')

fig.suptitle(f'Zero-excess diagnostics: {radius_label}', fontsize=14)
fig.tight_layout()
fig.savefig(PLOT_DIR / f'zero_excess_diagnostics_radius_{DIAG_RADIUS_DEF_ID}.png', dpi=180)
plt.show()

## Saved Figures

In [ ]:
print('Figures written to:', PLOT_DIR)
for path in sorted(PLOT_DIR.glob('*.png')):
    print(path.name)